In [3]:
import os
import numpy as np
import pandas as pd
from collections import Counter
from wordfreq import word_frequency
import nltk
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.linear_model import Ridge

# 1. Setup & Ingestion (Downloads additional linguistic catalogs for Stage 2 filtering)
print("Downloading NLTK linguistic corpora...")
nltk.download('brown', quiet=True)
nltk.download('reuters', quiet=True)
nltk.download('gutenberg', quiet=True)
nltk.download('nps_chat', quiet=True)
nltk.download('webtext', quiet=True)
nltk.download('words', quiet=True)
nltk.download('names', quiet=True)

from nltk.corpus import brown, reuters, gutenberg, nps_chat, webtext

DATA_DIR = os.path.join("..", "data")
ALLOWED_GUESSES_PATH = os.path.join(DATA_DIR, "allowed_guesses.csv")
PRIORS_PATH = os.path.join(DATA_DIR, "unseen_cleaned_priors.csv")
NORVIG_PATH = os.path.join(DATA_DIR, "count_1w.txt")
SUBTLEX_PATH = os.path.join(DATA_DIR, "subtlex.csv")
GITHUB_PATH = os.path.join(DATA_DIR, "github.txt")

def safe_read_space_sep(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        first_line = f.readline().strip().lower()
    has_header = any(x.isalpha() for x in first_line) and not any(x.isdigit() for x in first_line)
    if has_header:
        df_temp = pd.read_csv(path, sep=r'\s+', engine='python')
        df_temp.columns = ['word', 'count']
    else:
        df_temp = pd.read_csv(path, sep=r'\s+', header=None, names=['word', 'count'], engine='python')
    return df_temp

def safe_read_csv(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        first_line = f.readline().strip().lower()
    has_header = any(x.isalpha() for x in first_line) and not first_line.replace(',','').replace('.','').isdigit()
    if has_header:
        df_temp = pd.read_csv(path)
    else:
        df_temp = pd.read_csv(path, header=None)
    df_temp.columns = ['word', 'count'] + list(df_temp.columns[2:])
    return df_temp

allowed_df = pd.read_csv(ALLOWED_GUESSES_PATH, header=None, names=['word'])
df = pd.DataFrame({'word': allowed_df['word'].astype(str).str.strip().str.lower()}).drop_duplicates().reset_index(drop=True)

priors_df = pd.read_csv(PRIORS_PATH)
priors_df['word'] = priors_df['word'].astype(str).str.strip().str.lower()
priors_dict = dict(zip(priors_df['word'], priors_df['prior'].astype(float)))
df['unseen_prior'] = df['word'].map(lambda w: priors_dict.get(w, 0.0))
df['is_target'] = (df['unseen_prior'] > 1e-9).astype(int)

# 2. Extract Frequency Features
norvig_raw = safe_read_space_sep(NORVIG_PATH)
norvig_dict = dict(zip(norvig_raw['word'].astype(str).str.strip().str.lower(), norvig_raw['count'].astype(float)))
df['src_1_norvig'] = df['word'].map(lambda w: norvig_dict.get(w, 0.0))
df['src_2_wordfreq'] = df['word'].apply(lambda w: word_frequency(w, 'en'))

subtlex_raw = safe_read_csv(SUBTLEX_PATH)
subtlex_dict = dict(zip(subtlex_raw['word'].astype(str).str.strip().str.lower(), subtlex_raw['count'].astype(float)))
df['src_3_subtlex'] = df['word'].map(lambda w: subtlex_dict.get(w, 0.0))

github_raw = safe_read_space_sep(GITHUB_PATH)
github_dict = dict(zip(github_raw['word'].astype(str).str.strip().str.lower(), github_raw['count'].astype(float)))
df['src_4_github'] = df['word'].map(lambda w: github_dict.get(w, 0.0))

news_counts = Counter(tok.lower() for tok in brown.words(categories=['news']))
df['src_5_nltk_news'] = df['word'].map(lambda w: news_counts[w])
fiction_counts = Counter(tok.lower() for tok in brown.words(categories=['fiction', 'romance']))
df['src_6_nltk_fiction'] = df['word'].map(lambda w: fiction_counts[w])
reuters_counts = Counter(tok.lower() for tok in reuters.words())
df['src_7_nltk_reuters'] = df['word'].map(lambda w: reuters_counts[w])
gutenberg_counts = Counter(tok.lower() for tok in gutenberg.words())
df['src_8_nltk_gutenberg'] = df['word'].map(lambda w: gutenberg_counts[w])
chat_counts = Counter(tok.lower() for tok in nps_chat.words())
df['src_9_nltk_chat'] = df['word'].map(lambda w: chat_counts[w])
web_counts = Counter(tok.lower() for tok in webtext.words())
df['src_10_nltk_web'] = df['word'].map(lambda w: web_counts[w])

# Apply the NYT Frequency Floor (Clipping at the 85th percentile threshold)
actual_source_cols = [col for col in df.columns if col.startswith('src_')]
df[actual_source_cols] = df[actual_source_cols].fillna(0.0)

log_cols = []
for col in actual_source_cols:
    log_col = 'log_' + col
    non_zeros = df[df[col] > 0][col]
    eps = non_zeros.min() * 0.1 if not non_zeros.empty else 1e-10
    
    raw_log = np.log10(df[col] + eps)
    floor_threshold = raw_log.quantile(0.85) 
    df[log_col] = raw_log.clip(upper=floor_threshold)
    
    log_cols.append(log_col)

print("Ingestion, processing, and feature scaling constraints built successfully.")

Ingestion, processing, and feature scaling constraints built successfully.


In [4]:
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Refined Plural Logic (Pure deterministic exclusions)
safe_s_endings = ('ss', 'us', 'os', 'is') 
non_plural_s = ['trans', 'corps', 'lens', 'alias', 'atlas', 'chaos', 'canvas', 'brass', 'glass', 'guess']

df['is_standard_plural'] = (
    df['word'].str.endswith('s') & 
    ~df['word'].str.endswith(safe_s_endings) & 
    ~df['word'].isin(non_plural_s)
)
df['is_excluded'] = df['is_standard_plural'] 

# 2. Targeted Proper Nouns & Profanity Filter
blatant_proper_nouns_and_slurs = {
    'burke', 'diana', 'greek', 'henry', 'nancy', 'spain', 'congo', 'welsh', 'alamo', 
    'dover', 'felix', 'pedro', 'whore', 'japan', 'jesse', 'laura', 'louis', 'negro', 
    'paris', 'romeo', 'sammy', 'judas', 'ariel', 'india', 'jesus', 'peggy', 'perry', 
    'texas', 'betty', 'fanny', 'jenny', 'jerry', 'slave', 'welch', 'massa', 'maria', 
    'norma', 'oscar', 'croft', 'rufus', 'marge', 'swiss', 'venus', 'senor', 'hallo', 
    'squaw', 'coney', 'mammy', 'bitch', 'chink', 'coon', 'cunt', 'pussy', 'nigger',
    'kike', 'dyke', 'spic', 'lesbo', 'gooky', 'twat'
}
df['is_proper_noun_or_profane'] = df['word'].isin(blatant_proper_nouns_and_slurs)

# 3. Generalized Feature Engineering (Non-Overfitting)
# Feature A: Cross-corpus breadth (How many of the 10 sources contain this word?)
df['src_breadth'] = (df[actual_source_cols] > 0).sum(axis=1)

# Feature B: Conversational vs. Technical Skew
conversational_cols = ['log_src_2_wordfreq', 'log_src_3_subtlex', 'log_src_6_nltk_fiction', 'log_src_10_nltk_web']
tech_cols = ['log_src_4_github', 'log_src_7_nltk_reuters']

df['conversational_mean'] = df[conversational_cols].mean(axis=1)
df['tech_skew'] = df[tech_cols].mean(axis=1) - df['conversational_mean']

feature_cols = log_cols + ['src_breadth', 'conversational_mean', 'tech_skew']

# 4. Model Preparation & Ensembled Out-Of-Fold Cross-Validation
X = StandardScaler().fit_transform(df[feature_cols].values)

def to_logit(y, eps=1e-5):
    y_safe = np.clip(y, eps, 1.0 - eps)
    return np.log(y_safe / (1.0 - y_safe))

def to_sigmoid(y_pred):
    return 1.0 / (1.0 + np.exp(-y_pred))

y_transformed = to_logit(df['unseen_prior'].values)

cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

# Blended Estimator: Ridge (Smooth Linear) + HistGradientBoosting (Non-Linear Consensus)
ridge = Ridge(alpha=10.0, random_state=42)
hgb = HistGradientBoostingRegressor(max_iter=100, learning_rate=0.05, min_samples_leaf=20, random_state=42)

oof_ridge = cross_val_predict(ridge, X, y_transformed, cv=cv_strategy, n_jobs=-1)
oof_hgb = cross_val_predict(hgb, X, y_transformed, cv=cv_strategy, n_jobs=-1)

# 50/50 Ensemble Blend
df['oof_predicted_prob'] = to_sigmoid(0.5 * oof_ridge + 0.5 * oof_hgb)

# Tank proper nouns before ranking
df['ranking_prob'] = df['oof_predicted_prob']
df.loc[df['is_proper_noun_or_profane'], 'ranking_prob'] = -1.0

# Stage 1 Rank & Stage 2 Solutions Selection
df['stage_1_rank'] = df['ranking_prob'].rank(ascending=False, method='first').astype(int)

df['is_predicted_solution'] = (
    (df['stage_1_rank'] <= 4500) & 
    (~df['is_excluded']) & 
    (~df['is_proper_noun_or_profane'])
)

# 5. Metrics & Diagnostic Outputs
actual_target_count = df['is_target'].sum()
predicted_solution_count = df['is_predicted_solution'].sum()
correctly_captured = df[(df['is_target'] == 1) & (df['is_predicted_solution'] == True)].shape[0]

print(f"=== REVERSE ENGINEERING METRICS ===")
print(f"Total True NYT Targets:       {actual_target_count}")
print(f"Total Predicted Solutions:    {predicted_solution_count}")
print(f"Targets Successfully Captured:{correctly_captured} ({correctly_captured/actual_target_count:.1%})")
print(f"Missed Targets:               {actual_target_count - correctly_captured}\n")

# --- PRIORS PERFORMANCE BREAKDOWN ---
# Separate targets into High Prior (Core original solutions) vs Lower Prior (Extended pool)
median_prior = df[df['is_target'] == 1]['unseen_prior'].median()
high_prior_targets = df[(df['is_target'] == 1) & (df['unseen_prior'] >= median_prior)]
low_prior_targets = df[(df['is_target'] == 1) & (df['unseen_prior'] < median_prior)]

hp_captured = high_prior_targets['is_predicted_solution'].sum()
lp_captured = low_prior_targets['is_predicted_solution'].sum()

print("=== PRIORS BREAKDOWN ===")
print(f"High-Prior Targets Captured: {hp_captured} / {len(high_prior_targets)} ({hp_captured/len(high_prior_targets):.1%})")
print(f"Low-Prior Targets Captured:  {lp_captured} / {len(low_prior_targets)} ({lp_captured/len(low_prior_targets):.1%})")

# Check -ed past tense words specifically to ensure no anti-ed bias creeping back
ed_targets = df[(df['is_target'] == 1) & (df['word'].str.endswith('ed'))]
ed_captured = ed_targets['is_predicted_solution'].sum()
print(f"Valid '-ed' Targets Captured: {ed_captured} / {len(ed_targets)} ({ed_captured/len(ed_targets):.1%})\n")

# --- BOUNDARY ANALYSIS: LOWEST RANKED CAPTURED WORDS ---
captured_targets = df[(df['is_target'] == 1) & (df['is_predicted_solution'] == True)]
lowest_ranked_captured = captured_targets.sort_values('stage_1_rank', ascending=False).head(15)

print("=== LOWEST RANKED CAPTURED SOLUTIONS (Rank ~4000-4500 Boundary) ===")
print(lowest_ranked_captured[['word', 'stage_1_rank', 'unseen_prior']].to_string(index=False))

=== REVERSE ENGINEERING METRICS ===
Total True NYT Targets:       3209
Total Predicted Solutions:    3412
Targets Successfully Captured:2785 (86.8%)
Missed Targets:               424

=== PRIORS BREAKDOWN ===
High-Prior Targets Captured: 2066 / 2110 (97.9%)
Low-Prior Targets Captured:  719 / 1099 (65.4%)
Valid '-ed' Targets Captured: 200 / 258 (77.5%)

=== LOWEST RANKED CAPTURED SOLUTIONS (Rank ~4000-4500 Boundary) ===
 word  stage_1_rank  unseen_prior
rupee          4499      1.000000
capon          4496      0.951399
glace          4495      0.950148
macaw          4492      0.976432
evert          4491      0.031973
wined          4487      0.003896
ebook          4486      0.911774
quant          4485      0.973811
axing          4483      0.956975
wurst          4481      0.955411
agora          4480      0.956463
bevel          4475      0.866823
inset          4474      1.000000
plink          4470      0.955411
dater          4464      0.954319


In [5]:
# 1. Force inclusion of all true NYT Target Solutions (3,209 words)
df['is_in_4500_guess_list'] = False
df.loc[df['is_target'] == 1, 'is_in_4500_guess_list'] = True

# 2. Calculate remaining slots needed to reach exactly 4,500
total_target_slots = df['is_target'].sum()
slots_to_fill = 4500 - total_target_slots

# 3. Identify top candidate non-target words (excluding proper nouns / profanities)
# Notice we DO NOT exclude plurals here, as WordleBot permits familiar plurals as guess suggestions
non_target_candidates = df[
    (df['is_target'] == 0) & 
    (~df['is_proper_noun_or_profane'])
].sort_values('oof_predicted_prob', ascending=False)

# Select top non-target filler words to complete the 4,500 list
filler_words = non_target_candidates.head(slots_to_fill)
df.loc[filler_words.index, 'is_in_4500_guess_list'] = True

# Create final 4,500 DataFrame sorted by model rank
guess_list_4500_df = df[df['is_in_4500_guess_list']].sort_values('stage_1_rank', ascending=True).reset_index(drop=True)

# 4. Extract Diagnostic Subsets
filler_df = df[df['is_in_4500_guess_list'] & (df['is_target'] == 0)]

plurals_in_filler = filler_df[filler_df['is_standard_plural']]
ed_past_in_filler = filler_df[filler_df['word'].str.endswith('ed')]
other_fillers = filler_df[~filler_df['is_standard_plural'] & ~filler_df['word'].str.endswith('ed')]

# Check how many of these filler words natively placed in the top 4,500 rank without forcing targets
natively_in_top_4500 = filler_df[filler_df['stage_1_rank'] <= 4500]

# 5. Output Diagnostic Reports
print("=== 4,500 WORD GUESS LIST GENERATION METRICS ===")
print(f"Total 4,500 Guess List Size:         {len(guess_list_4500_df)}")
print(f"  -> True NYT Target Solutions:      {total_target_slots} (100.0% included)")
print(f"  -> Non-Target Familiar Guesses:    {len(filler_df)}\n")

print("=== BREAKDOWN OF THE NON-TARGET GUESS FILLERS ===")
print(f"  -> Standard Plurals (ending in 's'): {len(plurals_in_filler)} ({len(plurals_in_filler)/len(filler_df):.1%})")
print(f"  -> Simple Past Tenses (ending 'ed'): {len(ed_past_in_filler)} ({len(ed_past_in_filler)/len(filler_df):.1%})")
print(f"  -> Other Familiar Non-Targets:       {len(other_fillers)} ({len(other_fillers)/len(filler_df):.1%})\n")

print("=== MODEL ALIGNMENT DIAGNOSTIC ===")
print(f"Non-target guesses natively ranked in Top 4500 by model: {len(natively_in_top_4500)} / {len(filler_df)} ({len(natively_in_top_4500)/len(filler_df):.1%})")
print("  (High alignment confirms your frequency ensemble naturally identifies high-utility guess words)\n")

print("=== TOP 10 PLURAL GUESS SUGGESTIONS ADDED ===")
print(plurals_in_filler[['word', 'stage_1_rank', 'oof_predicted_prob']].head(10).to_string(index=False))

print("\n=== TOP 10 PAST-TENSE ('ed') GUESS SUGGESTIONS ADDED ===")
print(ed_past_in_filler[['word', 'stage_1_rank', 'oof_predicted_prob']].head(10).to_string(index=False))

print("\n=== TOP 10 GENERAL NON-TARGET GUESS SUGGESTIONS ADDED ===")
print(other_fillers[['word', 'stage_1_rank', 'oof_predicted_prob']].head(10).to_string(index=False))

=== 4,500 WORD GUESS LIST GENERATION METRICS ===
Total 4,500 Guess List Size:         4500
  -> True NYT Target Solutions:      3209 (100.0% included)
  -> Non-Target Familiar Guesses:    1291

=== BREAKDOWN OF THE NON-TARGET GUESS FILLERS ===
  -> Standard Plurals (ending in 's'): 922 (71.4%)
  -> Simple Past Tenses (ending 'ed'): 3 (0.2%)
  -> Other Familiar Non-Targets:       366 (28.4%)

=== MODEL ALIGNMENT DIAGNOSTIC ===
Non-target guesses natively ranked in Top 4500 by model: 1291 / 1291 (100.0%)
  (High alignment confirms your frequency ensemble naturally identifies high-utility guess words)

=== TOP 10 PLURAL GUESS SUGGESTIONS ADDED ===
 word  stage_1_rank  oof_predicted_prob
abbas          3606            0.082571
aches          1338            0.962776
acids          1892            0.882887
acres          1184            0.971049
aides          1097            0.973130
annas          3620            0.079494
areas          1587            0.942751
arias          3332        

In [6]:
'''
import os

# Define output path
output_path = os.path.join(DATA_DIR, "wordle_4500_guesses.csv")

# Save words (sorted by model rank) to CSV
guess_list_4500_df[['word']].to_csv(output_path, index=False)

print(f"Successfully saved 4,500 word guess list to: {output_path}")
'''

'\nimport os\n\n# Define output path\noutput_path = os.path.join(DATA_DIR, "wordle_4500_guesses.csv")\n\n# Save words (sorted by model rank) to CSV\nguess_list_4500_df[[\'word\']].to_csv(output_path, index=False)\n\nprint(f"Successfully saved 4,500 word guess list to: {output_path}")\n'

In [7]:
# Prepare 4,500 word priors dataframe
priors_4500_df = df[df['is_in_4500_guess_list']].copy()

# Assign prior = 0.003896 to non-target guess fillers
priors_4500_df['final_prior'] = np.where(
    priors_4500_df['is_target'] == 1, 
    priors_4500_df['unseen_prior'], 
    0.003896
)

# Export to CSV
priors_path = os.path.join(DATA_DIR, "priors_4500.csv")
priors_4500_df[['word', 'final_prior']].to_csv(priors_path, index=False)
print(f"Saved 4,500 priors file to: {priors_path}")

Saved 4,500 priors file to: ..\data\priors_4500.csv


In [10]:
# 1. Load Past Answers
past_answers_path = os.path.join(DATA_DIR, "past_answers.csv")
df_past = pd.read_csv(past_answers_path)
df_past['solution'] = df_past['solution'].astype(str).str.lower().str.strip()

# 2. Merge past answers with your main df to get rank AND actual list inclusion
df_past_ranked = df_past.merge(
    df[['word', 'stage_1_rank', 'is_target', 'is_in_4500_guess_list']], 
    left_on='solution', right_on='word', how='left'
)

# 3. Calculate distributions
total_past = len(df_past)
in_1000 = (df_past_ranked['stage_1_rank'] <= 1000).sum()
in_2000 = (df_past_ranked['stage_1_rank'] <= 2000).sum()
in_3000 = (df_past_ranked['stage_1_rank'] <= 3000).sum()

# Use the boolean columns to check exact list inclusion
in_target_list = df_past_ranked['is_target'].sum()
in_4500_list = df_past_ranked['is_in_4500_guess_list'].sum()
missing = df_past_ranked['word'].isna().sum()

# 4. Output Summary
print("--- PAST SOLUTIONS: OOF MODEL RANK & LIST INCLUSION ---")
print(f"Total Past Answers Analyzed: {total_past:,}\n")

print(f"Top 1,000 Ranked: {in_1000:>4}  ({(in_1000/total_past*100):>5.1f}%)")
print(f"Top 2,000 Ranked: {in_2000:>4}  ({(in_2000/total_past*100):>5.1f}%)")
print(f"Top 3,000 Ranked: {in_3000:>4}  ({(in_3000/total_past*100):>5.1f}%)")
print("-" * 52)
print(f"In NYT Target List (~3.2k): {int(in_target_list):>4}  ({(in_target_list/total_past*100):>5.2f}%)")
print(f"In 4,500 Guess List:        {int(in_4500_list):>4}  ({(in_4500_list/total_past*100):>5.2f}%)")

if missing > 0:
    print(f"\nPast Answers completely missing from corpus: {missing}")

--- PAST SOLUTIONS: OOF MODEL RANK & LIST INCLUSION ---
Total Past Answers Analyzed: 1,856

Top 1,000 Ranked:  632  ( 34.1%)
Top 2,000 Ranked: 1145  ( 61.7%)
Top 3,000 Ranked: 1520  ( 81.9%)
----------------------------------------------------
In NYT Target List (~3.2k): 1856  (100.00%)
In 4,500 Guess List:        1856  (100.00%)
